# ⚽ World Cup 2026 - Data Ingest
### Powered by Elasticsearch Serverless

This notebook ingests live 2026 World Cup data into Elasticsearch.

**What this notebook does:**
1. Fetches live 2026 match data from [openfootball/worldcup.json](https://github.com/openfootball/worldcup.json) - updated daily with real results
2. Normalises and enriches each match document
3. Bulk-indexes everything into your `wc2026_matches` index

Run it once before the hack night, then re-run any time you want fresher results.

---
> **Data source:** `openfootball/worldcup.json` - public domain, no API key, updated daily as results come in.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers
import json

print('✅ Imports ready')

## 1. Connect to Elastic Serverless

Find your endpoint and API key in:
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)

info = es.info()
print(f'✅ Connected to Elasticsearch {info["version"]["number"]}')

## 2. Fetch live 2026 match data

Pulls directly from the openfootball GitHub repo - no API key needed, public domain.
Matches without scores are upcoming fixtures; matches with scores are completed results.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json'

resp = requests.get(DATA_URL, timeout=15)
resp.raise_for_status()
raw = resp.json()

all_matches = raw['matches']
played   = [m for m in all_matches if 'score' in m]
upcoming = [m for m in all_matches if 'score' not in m]

print(f'✅ Fetched {len(all_matches)} total matches')
print(f'   {len(played)} completed results')
print(f'   {len(upcoming)} upcoming fixtures')
print()
print('Most recent results:')
for m in played[-5:]:
    ft = m['score']['ft']
    print(f"  {m['date']}  {m['team1']:20s} {ft[0]}–{ft[1]}  {m['team2']}  ({m['group']})")

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'wc2026_matches'

mapping = {
    'mappings': {
        'properties': {
            # Match identity
            'date':         {'type': 'date'},
            'round':        {'type': 'keyword'},
            'group':        {'type': 'keyword'},
            'stadium':      {'type': 'keyword'},
            'stage':        {'type': 'keyword'},   # 'group', 'round_of_32', 'round_of_16', 'quarter', 'semi', 'final'
            'status':       {'type': 'keyword'},   # 'played', 'upcoming'

            # Teams
            'team1':        {'type': 'keyword'},
            'team2':        {'type': 'keyword'},

            # Score (only on played matches)
            'score_ft1':    {'type': 'integer'},   # team1 full-time goals
            'score_ft2':    {'type': 'integer'},   # team2 full-time goals
            'score_ht1':    {'type': 'integer'},   # team1 half-time goals
            'score_ht2':    {'type': 'integer'},   # team2 half-time goals
            'total_goals':  {'type': 'integer'},
            'winner':       {'type': 'keyword'},   # team name or 'draw'
            'team1_win':    {'type': 'boolean'},
            'team2_win':    {'type': 'boolean'},

            # Goals (nested so individual scorers can be queried)
            'goals': {
                'type': 'nested',
                'properties': {
                    'scorer':   {'type': 'keyword'},
                    'minute':   {'type': 'keyword'},
                    'team':     {'type': 'keyword'},
                    'type':     {'type': 'keyword'},  # 'goal', 'penalty', 'own_goal'
                }
            },
        }
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')

es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest all matches

Both completed results and upcoming fixtures go in.
The enriched index can then answer questions like:
- *"When do France and Argentina play?"* (upcoming)
- *"What was the score when Brazil played Morocco?"* (played)
- *"Who has scored the most goals so far?"* (goals nested field)

In [ ]:
KNOCKOUT_ROUNDS = {'Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals',
                   'Match for third place', 'Final'}

def round_to_stage(round_name):
    mapping = {
        'Round of 32':          'round_of_32',
        'Round of 16':          'round_of_16',
        'Quarter-finals':       'quarter',
        'Semi-finals':          'semi',
        'Match for third place':'third_place',
        'Final':                'final',
    }
    if round_name in mapping:
        return mapping[round_name]
    return 'group'


def enrich(m):
    doc = {
        'date':    m['date'],
        'round':   m['round'],
        'group':   m.get('group', 'Knockout'),
        'stadium': m.get('ground', ''),
        'stage':   round_to_stage(m['round']),
        'team1':   m['team1'],
        'team2':   m['team2'],
    }

    if 'score' in m:
        ft1, ft2 = m['score']['ft']
        ht = m['score'].get('ht', [None, None])

        if ft1 > ft2:
            winner = m['team1']
        elif ft2 > ft1:
            winner = m['team2']
        else:
            winner = 'draw'

        doc.update({
            'status':      'played',
            'score_ft1':   ft1,
            'score_ft2':   ft2,
            'score_ht1':   ht[0],
            'score_ht2':   ht[1],
            'total_goals': ft1 + ft2,
            'winner':      winner,
            'team1_win':   winner == m['team1'],
            'team2_win':   winner == m['team2'],
        })

        # Flatten goals from both sides into a nested list
        goals = []
        for g in m.get('goals1', []):
            goals.append({
                'scorer': g['name'],
                'minute': g.get('minute', ''),
                'team':   m['team1'],
                'type':   'own_goal' if g.get('owngoal') else ('penalty' if g.get('penalty') else 'goal'),
            })
        for g in m.get('goals2', []):
            goals.append({
                'scorer': g['name'],
                'minute': g.get('minute', ''),
                'team':   m['team2'],
                'type':   'own_goal' if g.get('owngoal') else ('penalty' if g.get('penalty') else 'goal'),
            })
        doc['goals'] = goals

    else:
        doc['status'] = 'upcoming'

    return doc


docs = [enrich(m) for m in all_matches]
actions = [{'_index': INDEX, '_source': d} for d in docs]

success, errors = helpers.bulk(es, actions)
es.indices.refresh(index=INDEX)

print(f'✅ Indexed {success} documents ({len(errors)} errors)')
print(f'   Played matches:   {sum(1 for d in docs if d["status"] == "played")}')
print(f'   Upcoming fixtures:{sum(1 for d in docs if d["status"] == "upcoming")}')

## 5. Verify: quick sense-check queries

Just to confirm the data landed correctly.

In [ ]:
# How many documents? Status breakdown?
resp = es.search(index=INDEX, body={
    'size': 0,
    'aggs': {
        'by_status': {'terms': {'field': 'status'}},
        'by_group':  {'terms': {'field': 'group', 'size': 20}},
    }
})

print(f"Total documents: {resp['hits']['total']['value']}")
print()
print('By status:')
for b in resp['aggregations']['by_status']['buckets']:
    print(f"  {b['key']:10s} {b['doc_count']}")
print()
print('By group:')
for b in resp['aggregations']['by_group']['buckets']:
    print(f"  {b['key']:12s} {b['doc_count']} matches")

In [ ]:
# Show the 5 most recent completed results
resp = es.search(index=INDEX, body={
    'size': 5,
    'query': {'term': {'status': 'played'}},
    'sort': [{'date': {'order': 'desc'}}],
    '_source': ['date', 'team1', 'team2', 'score_ft1', 'score_ft2', 'group', 'winner']
})

print('Most recent results:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['date']}  {s['team1']:20s} {s['score_ft1']}–{s['score_ft2']}  {s['team2']:20s}  winner: {s['winner']}")

In [ ]:
# Show upcoming fixtures (next 5 by date)
resp = es.search(index=INDEX, body={
    'size': 5,
    'query': {'term': {'status': 'upcoming'}},
    'sort': [{'date': {'order': 'asc'}}],
    '_source': ['date', 'round', 'team1', 'team2', 'group', 'stadium']
})

print('Next upcoming fixtures:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['date']}  {s['team1']:20s} vs  {s['team2']:20s}  ({s['group']}) @ {s['stadium']}")

---
## ✅ Index loaded

Your `wc2026_matches` index is ready and verified.

That completes the data ingest. The next step lives outside this notebook - see **`agent_builder_guide.md`** for building the conversational agent on top of this index.

**To refresh with the latest results**, re-run cells 2-4 at any time.

---
## 💡 Data ingest extensions

Ideas for enriching the data you load into Elasticsearch:

### 🟢 Beginner
- **Ingest squad data** - `openfootball` also has `worldcup.squads.json` at the same base URL. Add a second `wc2026_squads` index with player names, positions, and caps
- **Ingest stadium data** - `worldcup.stadiums.json` has city, capacity, and country for all 16 venues
- **Re-run after each matchday** - the notebook takes ~30 seconds; results are updated daily on the source repo

### 🟡 Intermediate
- **Add a `wc2026_groups` index** - fetch `worldcup.groups.json` (same base URL) and ingest group standings
- **Combine 2026 with historical data** - merge this index with the historical `world_cup_matches` index for full World Cup history

### 🔴 Advanced
- **Schedule a refresh** - use Elastic Workflows to re-run the ingest automatically every day during the tournament
- **Live scores** - swap the openfootball source for `worldcup26.ir` (free, no key needed for some endpoints) and poll every few minutes during matches
- **Player embeddings** - generate vector embeddings from player bios and add `dense_vector` fields for semantic search